<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_20_Revision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chunking**

Problem

LLMs cannot process an entire PDF directly.

Example:

500-page PDF

Sending everything to the LLM:

Too many tokens
Expensive
Slow

Solution

Split document into smaller pieces.



In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
text=text = """
Generative Pre-trained Transformers (GPT) are a family of neural network models
that use the transformer architecture to create human-like text. They are trained
on massive amounts of data and can perform tasks like translation, summarization,
and question answering.

Large Language Models (LLMs) have a token limit, meaning they cannot process an
entire 500-page PDF directly. This is why we use chunking to break down the document
into smaller pieces, convert those pieces into embedding vectors, and store them
in a vector database like FAISS for quick retrieval.
"""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_text(text)

In [ ]:
chunks

think of a book instead of searching the entire book we just search for page 10,page 12,page 18

Limitations:
very small chunk:
lose context

very large chunks:
waste tokens

# **Embeddings**

Problem

Computers don't understand text.

Example:

What is GPT?

FAISS cannot search words.

It searches numbers.

SOlution:
COnvert text into vectors

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

Text

 ↓

Vector

 ↓

[0.23, -0.44, 0.81 ...]

Meaning becomes coordinates in vector space.

Limitation

Bad embedding model:

Bad retrieval

# **FAISS**

Problem

Imagine:

10000 chunks

Checking every chunk manually is impossible.

Solution

Store vectors inside FAISS.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np
index = faiss.IndexFlatL2(384)

index.add(
    np.array(embeddings).astype("float32")
)

In [ ]:
query = "What is GPT?"
query_embedding = embedding_model.encode([query]).astype("float32")

In [ ]:
distance,indicies=index.search(
    query_embedding,
    k=3
)

In [ ]:
print("Indices of matching chunks:", indicies)
print("Distances:", distance)

Intuition

FAISS asks:

Which vectors are closest
to the question vector?

Limitation

FAISS only understands vectors.

No deep understanding.

closest vector doesnt always mean best


# **L2 Distance**

In [ ]:
#used in
faiss.IndexFlatL2()

Intuition

Measure distance between vectors.

Smaller distance:

More similar
Limitation

Magnitude affects results.

# **Normalization**

Problem

Magnitude can mislead similarity.

In [ ]:
faiss.normalize_L2(embeddings)

Intuition

Convert vectors to:

Unit vectors

Only direction matters.

Why?

Semantic meaning is often encoded in direction.

# **RAG**

Retrieval-Augmented Generation (RAG) is a technique where relevant information is retrieved from an external knowledge source and provided to an LLM so that it can generate accurate, context-aware answers without requiring retraining.

**Why was RAG invented?**

Imagine we ask a model:

What is written on page 47 of my PDF?

The model has never seen your PDF.

So it has two choices:

Option 1

Say:

I don't know.
Option 2

Hallucinate:

Page 47 discusses neural networks.

even though it doesn't.

This is the core problem.

**Traditional LLM**

Question
   ↓
LLM
   ↓
Answer

The answer comes only from the model's training knowledge.

Example:

Question:
Who is the Prime Minister of India?

LLM:
Narendra Modi

The model already learned this during pretraining.

**But what about YOUR data?**

Suppose your PDF says:

Abrar is the captain of Alamdar Sports Club.

Ask:

Who is the captain of Alamdar Sports Club?

The LLM has never seen this.

So it cannot answer correctly on its own.

# **Understand this using our chatbot Example**

Suppose PDF contains:

Polymorphism allows one interface to have multiple forms.

User asks:

What is polymorphism?

step 1:
user question:  What is polymorphism?

step 2:
convert question into Embeddings

step 3:
FAISS search

step 4:
Retrieve chunks

step 5:
Build Context

step 6:
send to LLM

step 7:
LLM generates


**Important Question**

Where did the answer come from?

Was it from:

A
Model training knowledge

or

B
Retrieved chunks from the PDF

Answer:

B

The model is mostly acting as a reader and summarizer.

The knowledge came from retrieval.





# **Why is it called Retrieval-Augmented Generation?**

Two parts exist.

Retrieval
PDF
↓
Chunks
↓
Embeddings
↓
FAISS
↓
Relevant Chunks

This retrieves information.

Generation
Retrieved Chunks
↓
LLM
↓
Natural Language Answer

This generates the answer.

**Limitation**

Bad Retrieval = Bad answer

# **Cross Encoder Re-ranking**

**Problem**

FAISS similarity isn't always accurate.

Retrieved:

Chunk 7
Chunk 2
Chunk 10

Maybe:

Chunk 2

is actually best.

### **Solution**

**Cross Encoder.**

cross_encoder.predict(
    [
        (question, chunk)
    ]
)
### **Intuition**

Question and chunk enter together.

[Question]
+
[Chunk]

Deep comparison.

**Benefit**

Better ranking.

**Limitation**

Slow.